## Q1 Research v2 - Fashion-MNIST Dropout Experiment

The Week 5 experiment (`week5_Q1_dropout_research.ipynb`) is now the **pilot study**.
Its loss curves were still falling at epoch 10, so the models never overfit and dropout
had nothing to correct. This version is designed so the comparison can give a clear answer.

| | v1 (pilot study) | v2 (this notebook) |
|---|---|---|
| Training length | 10 epochs | up to 150 epochs, early stopping on validation loss (patience 15) |
| Model kept | final epoch | epoch with the lowest validation loss |
| Reported on | validation set | official test set (used once, at the end) |
| Network | 784-128-64-10 | 784-512-256-10 |
| Dropout rates | 0.0, 0.3 | 0.0, 0.2, 0.5 |
| Training sizes | 2,000, 10,000 | 1,000, 2,000, 5,000, 10,000 (nested) |
| Seeds | 3 | 10 (+2 separate pilot seeds) |
| Optimizer | SGD, lr 0.1 | SGD + momentum 0.9, lr tuned per dropout rate |
| Train accuracy | measured with dropout on | measured in eval mode (dropout off) |

**Order:** 1 setup, 2 learning-rate pilot (validation only), 3 freeze protocol,
4 final runs, 5 test set (once), 6 analysis and figures.

In [ ]:
# Setup
from pathlib import Path
import copy
import json
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision import datasets, transforms
from scipy import stats
from sklearn.metrics import f1_score, confusion_matrix, precision_recall_fscore_support

cwd = Path.cwd()
if (cwd / "Q1" / "notebooks").exists():
    Q1_DIR = cwd / "Q1"
elif cwd.name == "Q1":
    Q1_DIR = cwd
elif cwd.name == "notebooks" and cwd.parent.name == "Q1":
    Q1_DIR = cwd.parent
else:
    raise RuntimeError("Run this notebook from Q1/notebooks, Q1, or the repository root.")

DATA_DIR = Q1_DIR / "data"
SPLIT_DIR = DATA_DIR / "split_indices"
RESULTS_DIR = Q1_DIR / "results" / "raw" / "v2"     # v1 files in results/raw are left untouched
MODELS_DIR = Q1_DIR / "results" / "models" / "v2"
FIGURES_DIR = Q1_DIR / "figures"
CONFIG_DIR = Q1_DIR / "configs"
for folder in (RESULTS_DIR, MODELS_DIR, FIGURES_DIR, CONFIG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# CPU on purpose: for this small MLP it was ~5x faster than MPS on the M4
# (0.12 s vs 0.67 s per 10,000-image epoch), and CPU results are exactly repeatable
device = torch.device("cpu")
print("Device:", device)
print("PyTorch:", torch.__version__)

In [ ]:
# Load Fashion-MNIST. The official test set is loaded here but NOT used until Section 5.
full_train_dataset = datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=True, transform=transforms.ToTensor()
)
test_dataset = datasets.FashionMNIST(
    root=DATA_DIR, train=False, download=True, transform=transforms.ToTensor()
)
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Hold every image in memory as one tensor (much faster than a DataLoader for this size).
# Dividing by 255 gives exactly the same values as transforms.ToTensor().
X_all = full_train_dataset.data.float().div(255).unsqueeze(1)
y_all = full_train_dataset.targets.clone()
X_test = test_dataset.data.float().div(255).unsqueeze(1)
y_test = test_dataset.targets.clone()
for i in (0, 12345, 59999):
    assert torch.equal(X_all[i], full_train_dataset[i][0])
print("Training pool:", len(y_all), "| Official test set (held back):", len(y_test))

In [ ]:
# Same Week 1 seed-42 split as v1: 10,000 training indices and 5,000 validation indices
train_indices = np.load(SPLIT_DIR / "train_indices.npy").astype(int)
val_indices = np.load(SPLIT_DIR / "val_indices.npy").astype(int)
assert len(train_indices) == 10000 and len(val_indices) == 5000
assert len(set(train_indices)) == 10000 and len(set(val_indices)) == 5000
assert not set(train_indices).intersection(val_indices)

X_val, y_val = X_all[val_indices], y_all[val_indices]
y_val_np = y_val.numpy()
print("Validation:", len(y_val), "| Overlap with training:", len(set(train_indices) & set(val_indices)))

In [ ]:
# Experiment settings (v2)
TRAIN_SIZES = [1000, 2000, 5000, 10000]   # nested: each subset is train_indices[:n]
DROPOUT_RATES = [0.0, 0.2, 0.5]
SEEDS = [42, 123, 456, 789, 1011, 2024, 3141, 4242, 5555, 6789]
PILOT_SEEDS = [0, 1]                       # used only for the learning-rate pilot
PILOT_LRS = [0.003, 0.01, 0.03, 0.1]
PILOT_SIZES = [2000, 10000]
HIDDEN_SIZES = [512, 256]
BATCH_SIZE = 32
MOMENTUM = 0.9
MAX_EPOCHS = 150
PATIENCE = 15                              # epochs without a lower validation loss before stopping
SIMILAR_CLASSES = [0, 2, 4, 6]             # T-shirt/top, Pullover, Coat, Shirt
SHIRT = 6

assert not set(PILOT_SEEDS) & set(SEEDS)
print("Final runs:", len(TRAIN_SIZES) * len(DROPOUT_RATES) * len(SEEDS))
for n in TRAIN_SIZES:
    counts = np.bincount(y_all[train_indices[:n]].numpy(), minlength=10)
    print(f"n={n:>5}: images per class {counts.min()}-{counts.max()}")

In [ ]:
# MLP with configurable width; dropout follows each hidden ReLU (same placement as v1)
class MLPClassifier(nn.Module):
    def __init__(self, dropout=0.0, hidden_sizes=HIDDEN_SIZES):
        super().__init__()
        layers = [nn.Flatten()]
        in_features = 784
        for width in hidden_sizes:
            layers += [nn.Linear(in_features, width), nn.ReLU(), nn.Dropout(dropout)]
            in_features = width
        layers.append(nn.Linear(in_features, 10))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

print(MLPClassifier(0.5))

In [ ]:
# Evaluation helpers
loss_fn = nn.CrossEntropyLoss()

def predict(model, X, y, batch_size=5000):
    # Eval mode: dropout is OFF, so train and validation numbers are comparable
    model.eval()
    total_loss, preds = 0.0, []
    with torch.no_grad():
        for start in range(0, len(y), batch_size):
            xb, yb = X[start:start + batch_size].to(device), y[start:start + batch_size].to(device)
            outputs = model(xb)
            total_loss += loss_fn(outputs, yb).item() * len(yb)
            preds.append(outputs.argmax(dim=1).cpu())
    return total_loss / len(y), torch.cat(preds).numpy()

def error_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    in_group = np.isin(y_true, SIMILAR_CLASSES)
    group_errors = in_group & (y_true != y_pred)
    within = group_errors & np.isin(y_pred, SIMILAR_CLASSES)
    shirts = y_true == SHIRT
    return {
        "accuracy": 100 * np.mean(y_true == y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        # Same measure as v1: within-group mistakes / all images of the 4 classes
        "within_group_confusion_pct": 100 * within.sum() / in_group.sum(),
        # Of the 4 classes' mistakes, the share that land on another class in the group
        "within_group_error_share_pct": 100 * within.sum() / max(group_errors.sum(), 1),
        "shirt_recall_pct": 100 * np.mean(y_pred[shirts] == SHIRT),
        # Predicted shirts / real shirts (below 1 means the model avoids saying "Shirt")
        "shirt_prediction_ratio": (y_pred == SHIRT).sum() / shirts.sum(),
    }

In [ ]:
# Train one model with early stopping; returns the model restored to its best epoch
def train_one(train_size, dropout, seed, lr):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    # Same seed -> same initial weights and same batch order for every dropout rate
    model = MLPClassifier(dropout).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM)
    order_generator = torch.Generator().manual_seed(seed)

    subset = torch.as_tensor(train_indices[:train_size])
    X_train, y_train = X_all[subset].to(device), y_all[subset].to(device)
    y_train_np = y_train.cpu().numpy()

    history = []
    best_val_loss, best_state, best_epoch, bad_epochs = float("inf"), None, 0, 0
    diverged = False
    start = time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        order = torch.randperm(train_size, generator=order_generator)
        for b in range(0, train_size, BATCH_SIZE):
            batch = order[b:b + BATCH_SIZE]
            optimizer.zero_grad()
            loss = loss_fn(model(X_train[batch]), y_train[batch])
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(batch)
        if not np.isfinite(running_loss):
            diverged = True
            break

        train_loss, train_pred = predict(model, X_train, y_train)
        val_loss, val_pred = predict(model, X_val, y_val)
        history.append({
            "epoch": epoch,
            "train_loss_dropout_on": running_loss / train_size,
            "train_loss": train_loss,
            "train_accuracy": 100 * np.mean(train_pred == y_train_np),
            "val_loss": val_loss,
            "val_accuracy": 100 * np.mean(val_pred == y_val_np),
        })
        if val_loss < best_val_loss:
            best_val_loss, best_epoch, bad_epochs = val_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    info = {
        "best_epoch": best_epoch,
        "epochs_trained": len(history),
        "hit_max_epochs": len(history) == MAX_EPOCHS and bad_epochs < PATIENCE,
        "diverged": diverged,
        "runtime_sec": time.perf_counter() - start,
    }
    return model, pd.DataFrame(history), info

### 2. Learning-rate pilot (validation set only)

Dropout networks often prefer a different learning rate, so each dropout rate gets its own.
The pilot uses separate seeds (0 and 1) and is scored by the lowest validation loss reached.

In [ ]:
# Learning-rate pilot (safe to rerun: finished pilot runs are skipped)
pilot_file = RESULTS_DIR / "pilot_lr.csv"
pilot = pd.read_csv(pilot_file) if pilot_file.exists() else pd.DataFrame()

for dropout in DROPOUT_RATES:
    for lr in PILOT_LRS:
        for size in PILOT_SIZES:
            for seed in PILOT_SEEDS:
                pilot_id = f"lr{lr}_n{size}_drop{dropout:.1f}_seed{seed}"
                if not pilot.empty and pilot_id in pilot["pilot_id"].values:
                    continue
                _, history, info = train_one(size, dropout, seed, lr)
                ok = not info["diverged"] and len(history) > 0
                row = {
                    "pilot_id": pilot_id, "learning_rate": lr, "train_size": size,
                    "dropout": dropout, "seed": seed,
                    "best_val_loss": history["val_loss"].min() if ok else np.inf,
                    "best_val_accuracy": history["val_accuracy"].max() if ok else np.nan,
                    **info,
                }
                pilot = pd.concat([pilot, pd.DataFrame([row])], ignore_index=True)
                pilot.to_csv(pilot_file, index=False)
                print(f"Pilot {pilot_id}: best val loss {row['best_val_loss']:.4f}, "
                      f"best epoch {info['best_epoch']}")

pilot_scores = pilot.groupby(["dropout", "learning_rate"])["best_val_loss"].mean().unstack()
print("\nMean best validation loss (lower is better):")
print(pilot_scores.round(4))

selected_lrs = {f"{d:.1f}": float(pilot_scores.loc[d].idxmin()) for d in DROPOUT_RATES}
print("\nSelected learning rates:", selected_lrs)
for d, lr in selected_lrs.items():
    if lr in (min(PILOT_LRS), max(PILOT_LRS)):
        print(f"Note: dropout {d} chose lr={lr}, the edge of PILOT_LRS. "
              "Consider adding a value beyond it and rerunning the pilot before freezing.")

### 3. Freeze the protocol

Write the hypotheses **in your own words** before running this cell. They are saved with the
protocol, and the notebook refuses to continue if anything changes afterwards.

In [ ]:
# Hypotheses: edit these BEFORE the first time this cell runs
HYPOTHESES = [
    "H1: Without dropout, the gap between training and validation accuracy grows as the training set gets smaller.",
    "H2: Dropout reduces that gap at every training size.",
    "H3: Dropout lowers the within-group confusion rate for T-shirt/top, pullover, coat and shirt, "
    "and the reduction is larger at 10,000 images than at 2,000 images.",
    "H4: The test-accuracy gain from dropout depends on training-set size.",
]

config_path = CONFIG_DIR / "q1_dropout_protocol_v2.json"
protocol = {
    "version": 2,
    "dataset": "Fashion-MNIST",
    "split": "Week 1 seed-42 split; training subsets are train_indices[:n]; "
             "official test set evaluated once after all final runs",
    "architecture": [784, *HIDDEN_SIZES, 10],
    "dropout_rates": DROPOUT_RATES,
    "training_sizes": TRAIN_SIZES,
    "seeds": SEEDS,
    "pilot_seeds": PILOT_SEEDS,
    "pilot_learning_rates": PILOT_LRS,
    "selected_learning_rates": selected_lrs,
    "optimizer": "SGD",
    "momentum": MOMENTUM,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "model_selection": "epoch with the lowest validation loss",
    "device": str(device),
    "similar_classes": SIMILAR_CLASSES,
    "primary_metric": "test_accuracy",
    "secondary_metrics": [
        "test_macro_f1", "test_within_group_confusion_pct",
        "test_within_group_error_share_pct", "test_shirt_recall_pct", "generalization_gap",
    ],
    "hypotheses": HYPOTHESES,
    "analysis_plan": "For each training size and each dropout rate above 0, compare with dropout 0.0 "
                     "paired by seed (10 pairs): mean difference, 95% t confidence interval, paired "
                     "t-test, Holm correction across the 8 comparisons per metric. H3 compares the "
                     "dropout effect at 2,000 vs 10,000 with Welch's t-test. Every condition is reported.",
}
if config_path.exists():
    if json.loads(config_path.read_text()) != protocol:
        raise ValueError("Frozen v2 protocol already exists and differs. Check before running.")
else:
    config_path.write_text(json.dumps(protocol, indent=2))
print("Frozen protocol:", config_path)

### 4. Final runs (validation only)

Safe to rerun: finished runs are skipped. Each checkpoint is the lowest-validation-loss epoch.

In [ ]:
# Run and save the final experiments
results_file = RESULTS_DIR / "runs.csv"
history_file = RESULTS_DIR / "epoch_history.csv"
runs = pd.read_csv(results_file) if results_file.exists() else pd.DataFrame()
all_history = pd.read_csv(history_file) if history_file.exists() else pd.DataFrame()

for size in TRAIN_SIZES:
    for dropout in DROPOUT_RATES:
        lr = selected_lrs[f"{dropout:.1f}"]
        for seed in SEEDS:
            run_id = f"n{size}_drop{dropout:.1f}_seed{seed}"
            if not runs.empty and run_id in runs["run_id"].values:
                continue

            model, history, info = train_one(size, dropout, seed, lr)
            if info["diverged"] or history.empty:
                raise RuntimeError(f"{run_id} diverged. Revisit the learning-rate pilot.")

            best = history[history["epoch"] == info["best_epoch"]].iloc[0]
            _, val_pred = predict(model, X_val, y_val)
            row = {
                "run_id": run_id, "train_size": size, "dropout": dropout,
                "seed": seed, "learning_rate": lr, **info,
                "train_loss": best["train_loss"], "train_accuracy": best["train_accuracy"],
                "val_loss": best["val_loss"],
                **{f"val_{k}": v for k, v in error_metrics(y_val_np, val_pred).items()},
            }
            row["generalization_gap"] = row["train_accuracy"] - row["val_accuracy"]

            runs = pd.concat([runs, pd.DataFrame([row])], ignore_index=True)
            runs.to_csv(results_file, index=False)
            history.insert(0, "run_id", run_id)
            all_history = pd.concat([all_history, history], ignore_index=True)
            all_history.to_csv(history_file, index=False)
            torch.save(model.state_dict(), MODELS_DIR / f"{run_id}.pt")
            print(f"{run_id}: best epoch {info['best_epoch']}/{info['epochs_trained']}, "
                  f"val acc {row['val_accuracy']:.2f}, gap {row['generalization_gap']:.2f}, "
                  f"{info['runtime_sec']:.0f}s")

expected_runs = [f"n{n}_drop{d:.1f}_seed{s}" for n in TRAIN_SIZES for d in DROPOUT_RATES for s in SEEDS]
print("\nSaved runs:", len(runs), "/", len(expected_runs))
print("Runs that hit MAX_EPOCHS without early stopping:", int(runs["hit_max_epochs"].sum()))

### 5. Official test set (run once)

This only runs when all final runs exist and the protocol is unchanged. After the first run
the saved test results are loaded instead of recomputed.

In [ ]:
# Evaluate every saved checkpoint on the official test set
test_file = RESULTS_DIR / "test_results.csv"
runs = pd.read_csv(results_file)
missing = sorted(set(expected_runs) - set(runs["run_id"]))
assert not missing, f"Finish the final runs before using the test set. Missing: {missing[:5]}"
assert json.loads(config_path.read_text()) == protocol, "Protocol changed after freezing."

y_test_np = y_test.numpy()
if test_file.exists():
    test_results = pd.read_csv(test_file)
    print("Test set already evaluated: loaded saved results.")
else:
    rows = []
    for run in runs.itertuples():
        model = MLPClassifier(run.dropout).to(device)
        model.load_state_dict(torch.load(MODELS_DIR / f"{run.run_id}.pt"))
        test_loss, test_pred = predict(model, X_test, y_test)
        rows.append({"run_id": run.run_id, "test_loss": test_loss,
                     **{f"test_{k}": v for k, v in error_metrics(y_test_np, test_pred).items()}})
        cm = confusion_matrix(y_test_np, test_pred, labels=list(range(10)))
        np.savez_compressed(RESULTS_DIR / f"{run.run_id}_test_predictions.npz",
                            y_true=y_test_np, y_pred=test_pred, confusion=cm)
        p, r, f1, support = precision_recall_fscore_support(
            y_test_np, test_pred, labels=list(range(10)), zero_division=0
        )
        pd.DataFrame({"class": class_names, "precision": p, "recall": r,
                      "f1": f1, "support": support}).to_csv(
                          RESULTS_DIR / f"{run.run_id}_test_class_metrics.csv", index=False
                      )
    test_results = pd.DataFrame(rows)
    test_results.to_csv(test_file, index=False)
    print("Evaluated", len(test_results), "checkpoints on the test set.")

df = runs.merge(test_results, on="run_id")

### 6. Analysis

In [ ]:
# Condition averages over the 10 seeds
summary = df.groupby(["train_size", "dropout"], as_index=False).agg(
    runs=("seed", "count"),
    learning_rate=("learning_rate", "first"),
    test_accuracy_mean=("test_accuracy", "mean"),
    test_accuracy_std=("test_accuracy", "std"),
    test_macro_f1_mean=("test_macro_f1", "mean"),
    within_group_confusion_mean=("test_within_group_confusion_pct", "mean"),
    within_group_error_share_mean=("test_within_group_error_share_pct", "mean"),
    shirt_recall_mean=("test_shirt_recall_pct", "mean"),
    shirt_prediction_ratio_mean=("test_shirt_prediction_ratio", "mean"),
    generalization_gap_mean=("generalization_gap", "mean"),
    best_epoch_median=("best_epoch", "median"),
)
summary.to_csv(RESULTS_DIR / "summary.csv", index=False)
print(summary.round(3).to_string(index=False))

In [ ]:
# Paired comparisons: each dropout rate vs no dropout, same seed, same training size
def holm(p_values):
    p = np.asarray(p_values, dtype=float)
    adjusted = np.empty_like(p)
    running_max = 0.0
    for rank, i in enumerate(np.argsort(p)):
        running_max = max(running_max, min(1.0, (len(p) - rank) * p[i]))
        adjusted[i] = running_max
    return adjusted

def paired_effect(metric, size, dropout):
    base = df[(df.train_size == size) & (df.dropout == 0.0)].set_index("seed")[metric]
    treated = df[(df.train_size == size) & (df.dropout == dropout)].set_index("seed")[metric]
    diff = (treated - base).dropna()
    return diff, base.loc[diff.index], treated.loc[diff.index]

PAIRED_METRICS = [
    "test_accuracy", "test_macro_f1", "test_within_group_confusion_pct",
    "test_within_group_error_share_pct", "test_shirt_recall_pct", "generalization_gap",
]
rows = []
for metric in PAIRED_METRICS:
    for size in TRAIN_SIZES:
        for dropout in DROPOUT_RATES[1:]:
            diff, base, treated = paired_effect(metric, size, dropout)
            n = len(diff)
            half_width = stats.t.ppf(0.975, n - 1) * diff.std(ddof=1) / np.sqrt(n)
            rows.append({
                "metric": metric, "train_size": size, "dropout": dropout, "pairs": n,
                "mean_diff": diff.mean(),
                "ci95_low": diff.mean() - half_width, "ci95_high": diff.mean() + half_width,
                "p_value": stats.ttest_rel(treated, base).pvalue,
                "seeds_above_zero": int((diff > 0).sum()),
            })
paired = pd.DataFrame(rows)
paired["p_holm"] = paired.groupby("metric")["p_value"].transform(holm)
paired.to_csv(RESULTS_DIR / "paired_tests.csv", index=False)
print(paired.round(4).to_string(index=False))

In [ ]:
# H3: is the dropout effect different at 2,000 vs 10,000 images? (Welch's t-test on paired differences)
rows = []
for metric in ["test_accuracy", "test_within_group_confusion_pct"]:
    for dropout in DROPOUT_RATES[1:]:
        small, _, _ = paired_effect(metric, 2000, dropout)
        large, _, _ = paired_effect(metric, 10000, dropout)
        rows.append({
            "metric": metric, "dropout": dropout,
            "effect_at_2000": small.mean(), "effect_at_10000": large.mean(),
            "difference": large.mean() - small.mean(),
            "p_value": stats.ttest_ind(large, small, equal_var=False).pvalue,
        })
interaction = pd.DataFrame(rows)
interaction.to_csv(RESULTS_DIR / "size_interaction_tests.csv", index=False)
print(interaction.round(4).to_string(index=False))

In [ ]:
# Figure 1: test accuracy and dropout gain vs training size
COLORS = {0.0: "#555555", 0.2: "#1f77b4", 0.5: "#d62728"}

def ci95(values):
    values = np.asarray(values)
    return stats.t.ppf(0.975, len(values) - 1) * values.std(ddof=1) / np.sqrt(len(values))

def plot_metric_vs_size(ax, metric, ylabel):
    for dropout in DROPOUT_RATES:
        group = df[df.dropout == dropout].groupby("train_size")[metric]
        ax.errorbar(TRAIN_SIZES, group.mean().loc[TRAIN_SIZES],
                    yerr=[ci95(group.get_group(n)) for n in TRAIN_SIZES],
                    marker="o", capsize=3, color=COLORS[dropout], label=f"dropout {dropout}")
    ax.set_xscale("log")
    ax.set_xticks(TRAIN_SIZES, [f"{n:,}" for n in TRAIN_SIZES])
    ax.minorticks_off()
    ax.set_xlabel("Training images")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_metric_vs_size(axes[0], "test_accuracy", "Test accuracy (%)")
axes[0].legend()
axes[0].set_title(f"Test accuracy (mean ± 95% CI, {len(SEEDS)} seeds)")
for dropout in DROPOUT_RATES[1:]:
    gains = paired[(paired.metric == "test_accuracy") & (paired.dropout == dropout)]
    axes[1].errorbar(gains.train_size, gains.mean_diff,
                     yerr=[gains.mean_diff - gains.ci95_low, gains.ci95_high - gains.mean_diff],
                     marker="o", capsize=3, color=COLORS[dropout], label=f"dropout {dropout}")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_xscale("log")
axes[1].set_xticks(TRAIN_SIZES, [f"{n:,}" for n in TRAIN_SIZES])
axes[1].minorticks_off()
axes[1].set_xlabel("Training images")
axes[1].set_ylabel("Accuracy gain vs no dropout (points)")
axes[1].set_title("Dropout gain (paired by seed, 95% CI)")
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "q1v2_accuracy_vs_size.png", dpi=160)
plt.show()

In [ ]:
# Figure 2: overfitting gap and similar-class errors vs training size
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_metric_vs_size(axes[0], "generalization_gap", "Train acc - val acc (points)")
axes[0].set_title("Generalization gap at best epoch")
plot_metric_vs_size(axes[1], "test_within_group_confusion_pct", "Within-group confusion (%)")
axes[1].set_title("T-shirt / Pullover / Coat / Shirt mix-ups")
plot_metric_vs_size(axes[2], "test_shirt_recall_pct", "Shirt recall (%)")
axes[2].set_title("Shirt recall")
axes[0].legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "q1v2_overfitting_and_errors.png", dpi=160)
plt.show()

In [ ]:
# Figure 3: loss curves (mean over seeds; runs stop at different epochs, so later
# epochs average only the runs still training)
history = pd.read_csv(history_file).merge(runs[["run_id", "train_size", "dropout"]], on="run_id")
fig, axes = plt.subplots(1, len(TRAIN_SIZES), figsize=(4 * len(TRAIN_SIZES), 3.8), sharey=True)
for ax, size in zip(axes, TRAIN_SIZES):
    for dropout in DROPOUT_RATES:
        curves = history[(history.train_size == size) & (history.dropout == dropout)]
        by_epoch = curves.groupby("epoch")
        val = by_epoch["val_loss"]
        ax.plot(val.mean().index, val.mean(), color=COLORS[dropout], label=f"val, dropout {dropout}")
        ax.fill_between(val.mean().index, val.min(), val.max(), color=COLORS[dropout], alpha=0.15)
        ax.plot(by_epoch["train_loss"].mean(), color=COLORS[dropout], linestyle=":")
    ax.set_title(f"{size:,} training images")
    ax.set_xlabel("Epoch")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Loss (solid = validation, dotted = train, eval mode)")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "q1v2_loss_curves.png", dpi=160)
plt.show()

In [ ]:
# Figure 4: test confusion among the 4 similar classes (row %, summed over seeds)
sizes_to_show = [TRAIN_SIZES[0], TRAIN_SIZES[-1]]
fig, axes = plt.subplots(len(sizes_to_show), len(DROPOUT_RATES), figsize=(4 * len(DROPOUT_RATES), 8))
for row_axes, size in zip(axes, sizes_to_show):
    for ax, dropout in zip(row_axes, DROPOUT_RATES):
        cm_total = sum(
            np.load(RESULTS_DIR / f"n{size}_drop{dropout:.1f}_seed{seed}_test_predictions.npz")["confusion"]
            for seed in SEEDS
        )
        sub = cm_total[np.ix_(SIMILAR_CLASSES, SIMILAR_CLASSES)]
        row_pct = 100 * sub / cm_total[SIMILAR_CLASSES].sum(axis=1, keepdims=True)
        ax.imshow(row_pct, cmap="Blues", vmin=0, vmax=100)
        labels = [class_names[i] for i in SIMILAR_CLASSES]
        ax.set_xticks(range(4), labels, rotation=45, ha="right")
        ax.set_yticks(range(4), labels)
        ax.set_title(f"{size:,} images, dropout {dropout}")
        for r in range(4):
            for c in range(4):
                ax.text(c, r, f"{row_pct[r, c]:.0f}", ha="center", va="center", fontsize=8,
                        color="white" if row_pct[r, c] > 60 else "black")
for ax in axes[:, 0]:
    ax.set_ylabel("Actual")
for ax in axes[-1]:
    ax.set_xlabel("Predicted (% of actual class)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "q1v2_similar_classes.png", dpi=160)
plt.show()

### 7. Exploratory analysis (not pre-registered)

The pre-registered overfitting measure (train - val accuracy at the selected epoch) compares
no-dropout models picked around epoch 15 with dropout models picked around epoch 35-65, so it
does not show how much each model overfits as training continues. The two measures below were
added **after** seeing the results and are reported as exploratory. They use only the saved
training histories: nothing is retrained and the test set is not used.

- **Gap at a matched epoch:** train accuracy - val accuracy (both with dropout off) at epoch 20.
  Every one of the 120 runs trained for at least 24 epochs, so all runs are included.
- **Validation-loss rise:** validation loss at the last epoch trained minus the lowest validation
  loss. Early stopping always trains 15 epochs past the best epoch, so this measures how much
  the model got worse during those 15 epochs.

In [ ]:
# EXPLORATORY (not pre-registered): overfitting at a matched epoch and after the best epoch
MATCHED_EPOCH = 20
explore_history = pd.read_csv(history_file).merge(
    runs[["run_id", "train_size", "dropout", "seed", "best_epoch"]], on="run_id"
)
assert explore_history.groupby("run_id")["epoch"].max().min() >= MATCHED_EPOCH

matched = explore_history[explore_history["epoch"] == MATCHED_EPOCH].set_index("run_id")
best_rows = explore_history[explore_history["epoch"] == explore_history["best_epoch"]].set_index("run_id")
last_rows = explore_history.groupby("run_id").tail(1).set_index("run_id")
explore = best_rows[["train_size", "dropout", "seed"]].copy()
explore["gap_at_matched_epoch"] = matched["train_accuracy"] - matched["val_accuracy"]
explore["val_loss_rise_after_best"] = last_rows["val_loss"] - best_rows["val_loss"]
explore = explore.reset_index()

# Paired by seed, same layout as the pre-registered tests (lower values = less overfitting)
rows = []
for metric in ["gap_at_matched_epoch", "val_loss_rise_after_best"]:
    for size in TRAIN_SIZES:
        base = explore[(explore.train_size == size) & (explore.dropout == 0.0)].set_index("seed")[metric]
        for dropout in DROPOUT_RATES[1:]:
            treated = explore[(explore.train_size == size) & (explore.dropout == dropout)].set_index("seed")[metric]
            diff = (treated - base).dropna()
            n = len(diff)
            half_width = stats.t.ppf(0.975, n - 1) * diff.std(ddof=1) / np.sqrt(n)
            rows.append({
                "metric": metric, "train_size": size, "dropout": dropout, "pairs": n,
                "no_dropout_mean": base.mean(), "dropout_mean": treated.mean(),
                "mean_diff": diff.mean(),
                "ci95_low": diff.mean() - half_width, "ci95_high": diff.mean() + half_width,
                "p_value": stats.ttest_rel(treated.loc[diff.index], base.loc[diff.index]).pvalue,
                "seeds_below_zero": int((diff < 0).sum()),
            })
exploratory = pd.DataFrame(rows)
exploratory["p_holm"] = exploratory.groupby("metric")["p_value"].transform(holm)
exploratory.to_csv(RESULTS_DIR / "exploratory_overfitting_tests.csv", index=False)
print(exploratory.round(4).to_string(index=False))

# Figure 5 (exploratory): both overfitting measures vs training size
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, ylabel, title in [
    (axes[0], "gap_at_matched_epoch", "Train acc - val acc (points)",
     f"Generalization gap at epoch {MATCHED_EPOCH}"),
    (axes[1], "val_loss_rise_after_best", "Val loss increase",
     "Validation-loss rise in the 15 epochs after the best epoch"),
]:
    for dropout in DROPOUT_RATES:
        group = explore[explore.dropout == dropout].groupby("train_size")[metric]
        ax.errorbar(TRAIN_SIZES, group.mean().loc[TRAIN_SIZES],
                    yerr=[ci95(group.get_group(n)) for n in TRAIN_SIZES],
                    marker="o", capsize=3, color=COLORS[dropout], label=f"dropout {dropout}")
    ax.set_xscale("log")
    ax.set_xticks(TRAIN_SIZES, [f"{n:,}" for n in TRAIN_SIZES])
    ax.minorticks_off()
    ax.set_xlabel("Training images")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
axes[0].legend()
fig.suptitle("Exploratory (not pre-registered), mean ± 95% CI over 10 seeds", fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "q1v2_exploratory_overfitting.png", dpi=160)
plt.show()

In [ ]:
# Record package versions for this experiment
versions = {
    "pytorch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "torchvision": __import__("torchvision").__version__,
    "sklearn": __import__("sklearn").__version__,
    "scipy": __import__("scipy").__version__,
    "device": str(device),
}
(CONFIG_DIR / "q1_dropout_v2_environment.json").write_text(json.dumps(versions, indent=2))
print("Results:", RESULTS_DIR)
print("Figures:", FIGURES_DIR)
print("Protocol:", config_path)

### What to report in the paper

- **Methods:** v1 as the pilot study and why it motivated v2 (still-falling loss curves,
  best epoch 9-10, small train/val gap); the v2 table at the top of this notebook; the
  learning-rate pilot; early stopping; the test set used once.
- **Results:** `summary.csv` and Figure 1 (main result), `paired_tests.csv` (report the
  mean difference with its 95% CI and Holm-adjusted p for every comparison, including the
  ones that are not significant), Figure 2 and Figure 4 for the error analysis,
  `size_interaction_tests.csv` for H3.
- **Check before writing:** `hit_max_epochs` should be 0 (otherwise some runs were still
  improving at 150 epochs) and the generalization gap without dropout should be clearly
  larger at small training sizes. If it isn't, the models still aren't overfitting.